# Project D, Person 4: Validating the model

The brief asks for an LLM component "validated against ground truth". This notebook does that
for Person 3's forecasts and re-checks the headline numbers from Person 2's notebook a second way.
It doesn't rerun the model or the DCF. It reads what they saved and asks four questions:

1. Is the output usable (right shape, sensible values)?
2. Did the model ground its answers in the filing it was given?
3. Could it have seen the answers?
4. Do the headline numbers survive being computed a different way?

Every check uses one of the four tests from Lecture 1 (closed form, independent method, hand
calculation, sanity bounds), and Part 7 lists which one backs each number.

## Part 0: Data

In [1]:
import os
import re
import json
import numpy as np
import pandas as pd

DATA, LM, DCF = os.path.join("..", "data"), os.path.join("..", "lm", "output"), os.path.join("..", "dcf")
OUT = "output"
os.makedirs(OUT, exist_ok=True)

master = pd.read_csv(os.path.join(DATA, "master.csv")).set_index("ticker")
lm = pd.read_csv(os.path.join(DATA, "lm_forecasts.csv"))
detail = pd.read_csv(os.path.join(LM, "lm_forecasts_detailed.csv"))
probe = pd.read_csv(os.path.join(LM, "leak_probe.csv")).set_index("ticker")
run_log = json.load(open(os.path.join(LM, "run_log.json")))
base = pd.read_csv(os.path.join(DCF, "output", "baseline_forecasts.csv")).set_index("ticker")
vals = pd.read_csv(os.path.join(DCF, "output", "dcf_valuations.csv"))
quant = pd.read_csv(os.path.join(DCF, "backlog_quantified.csv")).set_index("ticker")

RUNS = ["AI full", "AI no MD&A"]
FLAGGED = ["BA", "DD", "FTV", "MMM"]     # same four firms Person 2 flags

print(f"firms in master     : {len(master)}")
print(f"forecast rows       : {len(lm)}  ({lm['label'].value_counts().to_dict()})")
print(f"detail rows         : {len(detail)}")
print(f"model served        : {run_log['model_served']}")
print(f"training cutoff     : {run_log['training_data_cutoff']}  (temperature {run_log['temperature']})")
print(f"prompt version      : {run_log['prompt_version']}  sha {run_log['prompt_sha256']}")

firms in master     : 53
forecast rows       : 106  ({'AI full': 53, 'AI no MD&A': 53})
detail rows         : 106
model served        : ['llama-3.3-70b-instruct']
training cutoff     : Dec 2023  (temperature 0.0)
prompt version      : v1.0  sha 9a51f8c2fcc61005


## Part 1: Is the output usable?

Every firm should have one answer per run, with no blanks, and the short file the DCF reads should
be the same numbers as the detailed file the model wrote.

In [2]:
# 1. one row per firm per run, nothing missing
counts = lm.groupby("label")["ticker"].nunique()
same_firms = all(set(lm.loc[lm["label"] == r, "ticker"]) == set(master.index) for r in RUNS)
print(f"firms per run       : {counts.to_dict()}")
print(f"same 53 as master   : {same_firms}")
print(f"missing values      : {lm[['growth', 'margin']].isna().sum().sum()}")

# 2. short file vs detailed file
d2 = detail.rename(columns={"revenue_growth_fy2025": "growth", "operating_margin_fy2025": "margin"})
both = lm.merge(d2[["ticker", "label", "growth", "margin"]], on=["ticker", "label"], suffixes=("", "_d"))
gap = max((both["growth"] - both["growth_d"]).abs().max(), (both["margin"] - both["margin_d"]).abs().max())
print(f"rows matched        : {len(both)} of {len(lm)}")
print(f"largest difference  : {gap:.1e}")

firms per run       : {'AI full': 53, 'AI no MD&A': 53}
same 53 as master   : True
missing values      : 0
rows matched        : 106 of 106
largest difference  : 0.0e+00


Sanity bounds. Industrial revenue rarely moves more than about 30% in a year and operating
margins sit well inside -20% to 50%. Anything outside those should be looked at by hand.

In [3]:
G_LO, G_HI = -0.30, 0.40     # growth bounds
M_LO, M_HI = -0.20, 0.50     # margin bounds

out_g = lm[(lm["growth"] < G_LO) | (lm["growth"] > G_HI)]
out_m = lm[(lm["margin"] < M_LO) | (lm["margin"] > M_HI)]
print(f"growth outside [{G_LO:.0%}, {G_HI:.0%}] : {len(out_g)}")
print(f"margin outside [{M_LO:.0%}, {M_HI:.0%}] : {len(out_m)}")
print()
print(lm.groupby("label")[["growth", "margin"]].describe().T.round(3).to_string())

growth outside [-30%, 40%] : 0
margin outside [-20%, 50%] : 0

label         AI full  AI no MD&A
growth count   53.000      53.000
       mean     0.051       0.055
       std      0.047       0.043
       min     -0.050      -0.050
       25%      0.020       0.035
       50%      0.050       0.050
       75%      0.065       0.080
       max      0.250       0.200
margin count   53.000      53.000
       mean     0.152       0.152
       std      0.079       0.079
       min     -0.100      -0.100
       25%      0.102       0.103
       50%      0.156       0.159
       75%      0.200       0.200
       max      0.445       0.445


In [4]:
# how often the model picks a round number (whole percent)
for r in RUNS:
    g = lm.loc[lm["label"] == r, "growth"]
    whole = (np.abs(g * 100 - np.round(g * 100)) < 1e-9).mean()
    print(f"{r:<11} growth forecasts that are a whole percent: {whole:.0%}")

print()
print(detail.groupby("label")["confidence"].value_counts().unstack(fill_value=0).to_string())

AI full     growth forecasts that are a whole percent: 87%
AI no MD&A  growth forecasts that are a whole percent: 96%

confidence  low  medium
label                  
AI full       9      44
AI no MD&A   21      32


Everything lines up and nothing is out of bounds. Two things to note. Most growth forecasts are whole
percents (5%, 2%, 8%), so the model is giving a rounded call rather than a precise estimate. And it
never says "high" confidence, only "medium" or "low".

## Part 2: Did the model quote the filing?

Every answer comes with a growth quote and a margin quote from the 10-K. Person 3's script checks
each one with an exact substring match after stripping punctuation and case. A single changed word
fails that test, so we check again a different way: break the quote into overlapping five-word
pieces and count how many appear in the filing. A quote with 90% or more of its pieces in the text
is treated as found.

The model saw the filing with the company name replaced by "the Company", so pieces containing
"company" are skipped. The full run saw Item 1 and Item 7; the no-MD&A run saw Item 1 only.

In [5]:
MAX_CHARS = 400_000     # same cap as lm/forecast.py
N = 5                   # words per piece
FOUND = 0.90            # share of pieces that must appear


def words(s):
    s = str(s).lower().replace("’", "'")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)          # 1,234 -> 1234
    return re.findall(r"[a-z0-9.%$]+", s)


def pieces(w, n=N):
    out = []
    for i in range(len(w) - n + 1):
        out.append(" ".join(w[i:i + n]))
    return out


def read(rel):
    with open(os.path.join("..", rel), encoding="utf-8", errors="ignore") as f:
        return f.read()[:MAX_CHARS]


source = {}
for t in master.index:
    item1 = read(master.loc[t, "item1_path"])
    item7 = read(master.loc[t, "item7_path"])
    source[(t, "AI no MD&A")] = set(pieces(words(item1)))
    source[(t, "AI full")] = set(pieces(words(item1 + " " + item7)))

rows = []
for _, r in detail.iterrows():
    for q in ["growth_quote", "margin_quote"]:
        p = [x for x in pieces(words(r[q])) if "company" not in x]
        hits = 0
        for x in p:
            if x in source[(r["ticker"], r["label"])]:
                hits = hits + 1
        share = hits / len(p) if p else np.nan
        rows.append({"ticker": r["ticker"], "label": r["label"], "quote": q.split("_")[0],
                     "script": bool(r[q + "_verified"]), "share": share,
                     "ours": bool(share >= FOUND), "text": r[q]})
qc = pd.DataFrame(rows)

print(f"quotes checked: {len(qc)}")
print()
print(qc.groupby(["label", "quote"])[["script", "ours"]].mean().round(3).to_string())
print()
print(f"script and our check agree on {(qc['script'] == qc['ours']).mean():.0%} of quotes")

quotes checked: 212

                   script   ours
label      quote                
AI full    growth   0.943  0.962
           margin   0.830  0.868
AI no MD&A growth   0.887  0.849
           margin   0.925  0.868

script and our check agree on 95% of quotes


In [6]:
# where the two checks disagree, and how close the quote came
diff = qc[qc["script"] != qc["ours"]]
print(diff[["ticker", "label", "quote", "script", "share"]].round(2).to_string(index=False))

ticker      label  quote  script  share
   MMM    AI full growth   False   0.96
    PH    AI full margin   False   0.92
   ROK    AI full margin   False   0.93
    BA AI no MD&A growth    True   0.86
   FTV AI no MD&A growth    True   0.67
     J AI no MD&A margin    True   0.83
   LMT AI no MD&A margin    True   0.75
   OSK AI no MD&A margin   False   0.95
   PWR AI no MD&A margin    True   0.88
   WAB AI no MD&A margin    True   0.88


The disagreements are borderline cases either way: a long quote with one changed word passes ours and
fails the script; a short quote with a slightly different number format passes the script and falls
just under our 90%. Neither check is "right"; together they bracket the answer.

What matters more is the quotes both checks reject. Sorting them by how much of the quote is in the
filing separates paraphrases from things that aren't there.

In [7]:
def kind(share):
    if share >= FOUND:
        return "in the filing"
    if share >= 0.5:
        return "close paraphrase"
    return "not in the filing"


qc["kind"] = qc["share"].apply(kind)
print(qc.groupby(["label", "kind"]).size().unstack(fill_value=0).to_string())
print()
bad = qc[qc["kind"] == "not in the filing"].sort_values("share")
for _, r in bad.iterrows():
    print(f"{r['ticker']:<5} {r['label']:<11} {r['quote']:<7} {r['share']:.2f}  {r['text'][:95]}")

kind        close paraphrase  in the filing  not in the filing
label                                                         
AI full                    7             97                  2
AI no MD&A                11             91                  4

EMR   AI full     margin  0.00  Adjusted EBITA Margin was 24.7 percent, an increase of 1.3 percentage points
JCI   AI no MD&A  growth  0.00  Total Building Solutions $19.4 7%
AXON  AI no MD&A  growth  0.14  Annual recurring revenue totaled $1.0 billion as of December 31, 2024
RTX   AI full     margin  0.17  Operating profit margin was 8.1% in 2024, with a FAS/CAS operating adjustment of $833 million.
GWW   AI no MD&A  growth  0.33  No single end customer accounted for more than 10% of total sales for the year ended December 3
RTX   AI no MD&A  margin  0.38  Our U.S. government sales were as follows: $32,246, $31,628, and $30,317 in 2024, 2023, and 202


Six quotes out of 212 aren't in the filing. Some are table fragments stitched into a sentence
(JCI's "Total Building Solutions $19.4 7%") or table rows rewritten as prose (RTX's government
sales). One is a true fact used for the wrong job (GWW's customer-concentration line offered as
support for a growth forecast). EMR's is the one to worry about: it quotes an adjusted EBITA margin
of 24.7%, but the 10-K says 25.8%.

Next we check whether the numbers inside each quote appear in the filing.

In [8]:
def is_num(w):
    return len(re.sub(r"\D", "", w)) >= 2      # skip single digits like "1" or "2"


def with_neighbour(w, i):
    """a number plus the word after it (or before it, at the end), e.g. '25.8 percent'"""
    return (w[i], w[i + 1]) if i + 1 < len(w) else (w[i - 1], w[i])


# a bare number like "24.7" turns up somewhere in almost any 10-K, so each number is
# checked together with its neighbouring word
num_rows = []
for _, r in detail.iterrows():
    txt = read(master.loc[r["ticker"], "item1_path"])
    if r["label"] == "AI full":
        txt = txt + " " + read(master.loc[r["ticker"], "item7_path"])
    sw = [x.strip(".") for x in words(txt)]
    pairs = set(zip(sw, sw[1:]))
    for q in ["growth_quote", "margin_quote"]:
        w = [x.strip(".") for x in words(r[q])]
        nums = [i for i, x in enumerate(w) if is_num(x)]
        missing = []
        for i in nums:
            p = with_neighbour(w, i)
            if p not in pairs:
                missing.append(" ".join(p))
        num_rows.append({"ticker": r["ticker"], "label": r["label"], "quote": q.split("_")[0],
                         "numbers": len(nums), "missing": "; ".join(missing)})
nc = pd.DataFrame(num_rows)
with_nums = nc[nc["numbers"] > 0]

print(f"quotes containing numbers    : {len(with_nums)} of {len(nc)}")
print(f"all numbers found in context : {(with_nums['missing'] == '').mean():.0%}")
print()
print(with_nums[with_nums["missing"] != ""].to_string(index=False))

quotes containing numbers    : 100 of 212
all numbers found in context : 94%

ticker      label  quote  numbers                              missing
   EMR    AI full margin        2                         24.7 percent
   LMT    AI full growth        4                               35% of
   RTX    AI full margin        3                8.1% in; $833 million
   JCI AI no MD&A growth        1                             $19.4 7%
   RTX AI no MD&A margin        6 $32246 $31628; $31628 and; $30317 in
   TDG AI no MD&A growth        2                          2019 levels


In [9]:
qc.drop(columns="text").merge(nc, on=["ticker", "label", "quote"]).to_csv(
    os.path.join(OUT, "quote_check.csv"), index=False)
print("saved output/quote_check.csv")

saved output/quote_check.csv


So the quotes are mostly real: 89% are in the filing (97% counting close paraphrases), and 94% of
the quotes that cite numbers have every number in the filing with the same word next to it. The
misses are the same handful as above, plus two where the model reworded the text around a number
(LMT, TDG). The quotes are a check on the model, not
an input to the DCF, so a bad quote doesn't change any forecast. But the memo shouldn't repeat a
model quote without opening the filing first.

### Does the model's read of backlog match ours?

The model was asked whether the 10-K discloses backlog. We have two flags of our own: the keyword
flag from Person 1 (`backlog_mentioned`, the 38/15 split) and Person 2's stricter rule that the 10-K
puts a number on the change (`quantified_backlog`, 29/24). The stricter rule was written before
anyone looked at the model's answer.

In [10]:
full = detail[detail["label"] == "AI full"].set_index("ticker")
bk = pd.DataFrame({"model says disclosed": full["backlog_disclosed"],
                   "keyword flag": master["backlog_mentioned"],
                   "quantified rule": quant["quantified_backlog"]})

print(pd.crosstab(bk["model says disclosed"], bk["keyword flag"]).to_string())
print()
print(pd.crosstab(bk["model says disclosed"], bk["quantified rule"]).to_string())
print()
miss = bk[bk["keyword flag"] & ~bk["model says disclosed"]].index.tolist()
print(f"keyword yes, model no: {miss}")

keyword flag          False  True 
model says disclosed              
False                    15      9
True                      0     29

quantified rule       False  True 
model says disclosed              
False                    24      0
True                      0     29

keyword yes, model no: ['CARR', 'CMI', 'DD', 'IEX', 'IR', 'ITT', 'ITW', 'LECO', 'WSO']


The model's answer matches the quantified rule on all 53 firms. Two methods built separately (a regex
rule and the model reading the text) land on the same 29 firms, which is good evidence both are
picking up the same thing. The nine firms the keyword flag adds (CARR, CMI, DD, IEX, IR, ITT, ITW,
LECO, WSO) mention backlog without giving a number for it, and the model treats those as "none
disclosed".

## Part 3: Could the model have seen the answers?

FY2025 results came out in early 2026. Llama 3.3's training data stops in December 2023 (Meta's model
card), so it can't have read them. Person 3 also asked the model directly, with no filing, what it
remembered about each firm's FY2025.

In [11]:
print(f"firms where the model claims to know FY2025 : {probe['knows_actual_fy2025'].sum()} of {len(probe)}")
print(f"recalled growth within 1pp of actual         : {probe['recall_within_1pp'].sum()}")
print()
print("latest period the model says it knows:")
print(probe["latest_period_known"].value_counts().to_string())

firms where the model claims to know FY2025 : 0 of 53
recalled growth within 1pp of actual         : 0

latest period the model says it knows:
latest_period_known
FY2022    50
FY2024     3


It knows nothing about FY2025. Three firms (FIX, J, PH) get "FY2024" as the latest period it knows,
which is a little odd for a model with a 2023 cutoff, but it recalled no numbers for any of them.

One more check that doesn't rely on the model telling the truth: a model that had seen the answers
would land very close to actual far more often than the benchmarks do.

In [12]:
CLOSE = 0.005    # within half a point of actual growth

act = base["actual_growth"]
fc = {"3-yr average": base["base_growth"], "last year": base["last_growth"]}
for r in RUNS:
    fc[r] = lm[lm["label"] == r].set_index("ticker")["growth"].reindex(base.index)

for name, g in fc.items():
    print(f"{name:<13} within {CLOSE:.1%} of actual: {((g - act).abs() < CLOSE).sum():>2} of 53")

3-yr average  within 0.5% of actual:  5 of 53
last year     within 0.5% of actual:  7 of 53
AI full       within 0.5% of actual:  5 of 53
AI no MD&A    within 0.5% of actual:  7 of 53


The model is no closer to the answers than repeating last year is. There's no sign it saw them.

Person 2 found and fixed one leak on our side: for nine firms with non-December fiscal years, the
filing script had picked up the FY2025 10-K instead of FY2024, so the model was reading the answers.
Those nine were re-extracted and rerun (commits f91d5ae, 5708a93, dc488bd). The check below
confirms none of their forecasts now equal the actual.

In [13]:
NON_DEC = ["DE", "DY", "EMR", "J", "JCI", "NDSN", "PH", "ROK", "TDG"]
g = fc["AI full"].loc[NON_DEC]
chk = pd.DataFrame({"AI full": g, "actual": act.loc[NON_DEC], "miss": (g - act.loc[NON_DEC]).abs()})
print(chk.round(3).to_string())
print()
print(f"any forecast within 0.5pp of actual : {(chk['miss'] < CLOSE).any()}")
for name in ["AI full", "3-yr average", "last year"]:
    e = (fc[name] - act).abs()
    print(f"mean miss, {name:<12} these 9 vs other 44 : {e.loc[NON_DEC].mean():.1%} vs {e.drop(NON_DEC).mean():.1%}")

        AI full  actual   miss
ticker                        
DE       -0.050  -0.117  0.067
DY        0.080   0.126  0.046
EMR       0.045   0.030  0.015
J         0.060   0.046  0.014
JCI       0.050   0.028  0.022
NDSN      0.020   0.038  0.018
PH        0.050  -0.004  0.054
ROK       0.050   0.009  0.041
TDG       0.080   0.112  0.032

any forecast within 0.5pp of actual : False
mean miss, AI full      these 9 vs other 44 : 3.4% vs 5.1%
mean miss, 3-yr average these 9 vs other 44 : 6.5% vs 6.5%
mean miss, last year    these 9 vs other 44 : 5.1% vs 5.7%


None of the nine is suspiciously close. The model does better on these nine (3.4% against 5.1%) while
the three-year average does no better on them, which is worth a sentence in the limitations. With nine
firms it can easily be luck, and their filings are the fiscal-2024 10-Ks, filed before their FY2025
results existed.

## Part 4: Accuracy, recomputed from the raw files

Person 2's notebook scores the forecasts. Here the same numbers are rebuilt straight from
`master.csv` and `lm_forecasts.csv` instead of the DCF notebook's tables, so a mistake in that
notebook's data handling would show up as a mismatch.

In [14]:
# actuals taken from master.csv, not from the DCF notebook's output
act_g = master["actual_revenue_growth"].reindex(base.index)
act_m = master["actual_margin"].reindex(base.index)
print(f"actual growth, master vs DCF output: largest gap {(act_g - act).abs().max():.1e}")

mar = {"3-yr average": base["base_margin"], "last year": base["last_margin"]}
for r in RUNS:
    mar[r] = lm[lm["label"] == r].set_index("ticker")["margin"].reindex(base.index)

ex = ~base.index.isin(FLAGGED)
bkl = base["backlog"]
acc = {}
for name in fc:
    eg = (fc[name] - act_g).abs()
    em = (mar[name] - act_m).abs()
    acc[name] = {"growth MAE, all": eg.mean(), "growth MAE, backlog": eg[bkl].mean(),
                 "growth MAE, no backlog": eg[~bkl].mean(), "growth MAE, ex-flagged": eg[ex].mean(),
                 "margin MAE, all": em.mean()}
acc = pd.DataFrame(acc)
print(acc.round(4).to_string())

actual growth, master vs DCF output: largest gap 0.0e+00
                        3-yr average  last year  AI full  AI no MD&A
growth MAE, all               0.0651     0.0561   0.0477      0.0527
growth MAE, backlog           0.0747     0.0659   0.0517      0.0579
growth MAE, no backlog        0.0410     0.0314   0.0377      0.0395
growth MAE, ex-flagged        0.0542     0.0498   0.0452      0.0482
margin MAE, all               0.0282     0.0187   0.0179      0.0178


In [15]:
# Person 2's printed values (dcf notebook, Part 3)
P2 = {"3-yr average": 0.0651, "last year": 0.0561, "AI full": 0.0477, "AI no MD&A": 0.0527}
for k, v in P2.items():
    print(f"{k:<13} ours {acc.loc['growth MAE, all', k]:.4f}   notebook {v:.4f}"
          f"   agree {abs(acc.loc['growth MAE, all', k] - v) < 0.00005}")

# and the forecasts the DCF used are the ones the model wrote
used = vals[vals["scenario"] == "AI full"].set_index("ticker")["growth"]
print(f"\nDCF used the model's growth for every firm: {np.allclose(used.loc[base.index], fc['AI full'])}")

3-yr average  ours 0.0651   notebook 0.0651   agree True
last year     ours 0.0561   notebook 0.0561   agree True
AI full       ours 0.0477   notebook 0.0477   agree True
AI no MD&A    ours 0.0527   notebook 0.0527   agree True

DCF used the model's growth for every firm: True


### How the edge is spread across firms

A lower average can come from beating the benchmark on most firms or from avoiding a few big misses.
Counting wins firm by firm tells them apart.

In [16]:
eg = {k: (v - act_g).abs() for k, v in fc.items()}
for other in ["3-yr average", "last year", "AI no MD&A"]:
    w = eg["AI full"] < eg[other]
    print(f"AI full beats {other:<13} on {w.sum():>2} of 53 firms "
          f"(backlog {w[bkl].sum()} of {bkl.sum()}, no backlog {w[~bkl].sum()} of {(~bkl).sum()})")

print()
gap = (eg["3-yr average"] - eg["AI full"]).sort_values()
print("largest wins for the model over the 3-yr average (points of growth error):")
print((gap.tail(6)[::-1] * 100).round(1).to_string())
print("\nlargest losses:")
print((gap.head(4) * 100).round(1).to_string())

AI full beats 3-yr average  on 27 of 53 firms (backlog 21 of 38, no backlog 6 of 15)
AI full beats last year     on 21 of 53 firms (backlog 16 of 38, no backlog 5 of 15)
AI full beats AI no MD&A    on 18 of 53 firms (backlog 12 of 38, no backlog 6 of 15)

largest wins for the model over the 3-yr average (points of growth error):
ticker
DD      24.7
FTV     15.8
XYL     15.8
OSK     14.9
TEX      9.9
CARR     9.1

largest losses:
ticker
FIX    -13.7
GNRC    -9.5
AXON    -7.5
PWR     -5.8


The model beats the three-year average on only 27 of 53 firms, close to a coin flip. Its lower
average comes from a handful of large misses it avoids. The two biggest are flagged firms: for DD and
FTV the baseline averages over a restated FY2022 and forecasts -24% and -14%, while the model reads
the filing and calls +3% and +1.5%. That's a real advantage, but it's a different one from reading
backlog. The next four (XYL, OSK, TEX, CARR) are backlog firms where the baseline extrapolated an
acquisition or a boom year.

## Part 5: Is the edge more than luck?

Person 2's notebook runs a paired bootstrap with a loop and seed 0. This version draws all 10,000
resamples at once as an index matrix, uses seed 1, and adds the ex-flagged groups the notebook doesn't
bootstrap. If the two versions disagree beyond a few hundredths of a point, one of them is wrong.

In [17]:
DRAWS, SEED = 10_000, 1


def boot(diff, draws=DRAWS, seed=SEED):
    """mean of diff and 95% range, firms resampled with replacement"""
    d = np.asarray(diff)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(draws, len(d)))
    means = d[idx].mean(axis=1)
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


GROUPS = {"all firms": base.index == base.index, "backlog": bkl.to_numpy(),
          "no backlog": (~bkl).to_numpy(), "ex-flagged": ex, "backlog ex-flagged": bkl.to_numpy() & ex}

rows = []
for other in ["3-yr average", "last year", "AI no MD&A"]:
    for gname, mask in GROUPS.items():
        mean, lo, hi = boot((eg["AI full"] - eg[other])[mask])
        verdict = "reliably better" if hi < 0 else ("reliably worse" if lo > 0 else "not distinguishable")
        rows.append({"vs": other, "group": gname, "n": int(mask.sum()), "gap (pp)": mean * 100,
                     "lo": lo * 100, "hi": hi * 100, "verdict": verdict})
bt = pd.DataFrame(rows)
print(bt.round(2).to_string(index=False))

          vs              group  n  gap (pp)    lo    hi             verdict
3-yr average          all firms 53     -1.74 -3.56 -0.02     reliably better
3-yr average            backlog 38     -2.30 -4.55 -0.21     reliably better
3-yr average         no backlog 15     -0.33 -3.34  2.36 not distinguishable
3-yr average         ex-flagged 49     -0.90 -2.48  0.66 not distinguishable
3-yr average backlog ex-flagged 36     -1.63 -3.56  0.25 not distinguishable
   last year          all firms 53     -0.84 -2.27  0.47 not distinguishable
   last year            backlog 38     -1.42 -3.25  0.22 not distinguishable
   last year         no backlog 15      0.63 -0.98  2.32 not distinguishable
   last year         ex-flagged 49     -0.46 -1.77  0.77 not distinguishable
   last year backlog ex-flagged 36     -0.98 -2.58  0.52 not distinguishable
  AI no MD&A          all firms 53     -0.50 -1.22  0.18 not distinguishable
  AI no MD&A            backlog 38     -0.62 -1.53  0.18 not distinguishable

In [18]:
# the notebook's numbers for the same comparisons (seed 0, loop version)
P2_BOOT = {("3-yr average", "all firms"): (-3.57, -0.02), ("3-yr average", "backlog"): (-4.53, -0.23),
           ("last year", "all firms"): (-2.23, 0.44), ("last year", "backlog"): (-3.20, 0.19)}
for (o, g), (lo, hi) in P2_BOOT.items():
    r = bt[(bt["vs"] == o) & (bt["group"] == g)].iloc[0]
    print(f"vs {o:<13} {g:<10} ours [{r['lo']:+.2f}, {r['hi']:+.2f}]  notebook [{lo:+.2f}, {hi:+.2f}]")

vs 3-yr average  all firms  ours [-3.56, -0.02]  notebook [-3.57, -0.02]
vs 3-yr average  backlog    ours [-4.55, -0.21]  notebook [-4.53, -0.23]
vs last year     all firms  ours [-2.27, +0.47]  notebook [-2.23, +0.44]
vs last year     backlog    ours [-3.25, +0.22]  notebook [-3.20, +0.19]


The two bootstraps agree to within about a tenth of a point, so the notebook's ranges are right. But
the ex-flagged rows change the story. With all 53 firms the model is reliably better than the
three-year average, and the upper end of that range sits just below zero. Take out BA, DD, FTV and
MMM and the gap halves to about 0.9 points and is no longer distinguishable from luck, even on
backlog firms. Against "last year" the model is never reliably better in any group.

So the honest headline is: the model clearly helps where the three-year average is mechanically broken
(restated or loss-making histories), helps a little on backlog firms, and isn't distinguishable from
simply repeating last year.

In [19]:
# does the model know when it's guessing?
conf = detail[detail["label"] == "AI full"].set_index("ticker")["confidence"]
print(pd.DataFrame({"miss": eg["AI full"], "confidence": conf}).groupby("confidence")["miss"]
      .agg(["count", "mean", "median"]).round(3).to_string())

# is it mostly repeating last year?
corr_last = np.corrcoef(fc["AI full"], fc["last year"])[0, 1]
corr_base = np.corrcoef(fc["AI full"], fc["3-yr average"])[0, 1]
m_last = ((mar["AI full"] - mar["last year"]).abs() < 0.005).mean()
print(f"\ncorrelation of AI growth with last year {corr_last:.2f}, with 3-yr average {corr_base:.2f}")
print(f"AI margin within 0.5pp of FY2024 margin: {m_last:.0%} of firms")

            count   mean  median
confidence                      
low             9  0.071   0.036
medium         44  0.043   0.030

correlation of AI growth with last year 0.76, with 3-yr average 0.62
AI margin within 0.5pp of FY2024 margin: 64% of firms


Low-confidence answers miss by more on average (driven by a couple of big misses; the medians are
close), so the confidence label carries a little information. The model's growth calls track last
year's growth more than the three-year average, and about two thirds of its margins sit within half a
point of FY2024. That's consistent with the finding above: on margin, the model is mostly repeating
last year, which happens to be a good forecast.

## Part 6: Valuation errors, recomputed in closed form

Person 2's `dcf()` builds each valuation with a loop. With a straight-line fade, the whole DCF can be
written as array arithmetic over all 212 rows at once, which is a structurally different calculation.
If it reproduces every EV in `dcf_valuations.csv`, the valuation errors are right.

In [20]:
WACC, G_TERM, TAX, ROIC, YEARS = 0.088, 0.025, 0.25, 0.20, 5   # Person 2's base case

v = vals.copy()
r0 = master["revenue_fy_minus_1"].reindex(v["ticker"]).to_numpy()
g1 = v["growth"].to_numpy()[:, None]
m = v["margin"].to_numpy()[:, None]

t = np.arange(1, YEARS + 1)[None, :]
path = g1 + (G_TERM - g1) * (t - 1) / (YEARS - 1)          # fade, one row per valuation
rev = r0[:, None] * np.cumprod(1 + path, axis=1)
fcf = rev * m * (1 - TAX) * (1 - np.maximum(path, 0) / ROIC)
disc = (1 + WACC) ** -t
tv = rev[:, -1] * (1 + G_TERM) * m[:, 0] * (1 - TAX) * (1 - G_TERM / ROIC) / (WACC - G_TERM)
ev = (fcf * disc).sum(axis=1) + tv * disc[0, -1]

rel = np.abs(ev / v["ev"].to_numpy() - 1)
print(f"valuations recomputed : {len(ev)}")
print(f"largest relative gap  : {rel.max():.1e}")

valuations recomputed : 212
largest relative gap  : 2.3e-15


### Hand check on one firm

Caterpillar under the model's forecast, shrunk to numbers you could do on paper: revenue in
$ billions and five rounded steps.

In [21]:
row = v[(v["ticker"] == "CAT") & (v["scenario"] == "AI full")].iloc[0]
r, g, mg = master.loc["CAT", "revenue_fy_minus_1"] / 1e9, row["growth"], row["margin"]
print(f"CAT FY2024 revenue ${r:.2f}bn, AI growth {g:.1%}, AI margin {mg:.1%}")

total = 0.0
rev_t = r
for yr in range(1, 6):
    gt = g + (G_TERM - g) * (yr - 1) / 4
    rev_t = rev_t * (1 + gt)
    cash = rev_t * mg * 0.75 * (1 - max(gt, 0) / 0.20)
    pv = cash / 1.088 ** yr
    total = total + pv
    print(f"  FY{2024 + yr}  growth {gt:6.2%}  revenue {rev_t:6.2f}  FCF {cash:5.2f}  PV {pv:5.2f}")

tv_pv = rev_t * 1.025 * mg * 0.75 * (1 - 0.025 / 0.20) / (0.088 - 0.025) / 1.088 ** 5
print(f"PV of 5 years {total:.2f} + PV of terminal {tv_pv:.2f} = EV ${total + tv_pv:.2f}bn")
print(f"dcf_valuations.csv says ${row['ev'] / 1e9:.2f}bn")

CAT FY2024 revenue $64.81bn, AI growth -1.0%, AI margin 20.0%
  FY2025  growth -1.00%  revenue  64.16  FCF  9.62  PV  8.85
  FY2026  growth -0.12%  revenue  64.08  FCF  9.61  PV  8.12
  FY2027  growth  0.75%  revenue  64.56  FCF  9.32  PV  7.24
  FY2028  growth  1.62%  revenue  65.61  FCF  9.04  PV  6.45
  FY2029  growth  2.50%  revenue  67.25  FCF  8.83  PV  5.79
PV of 5 years 36.45 + PV of terminal 94.20 = EV $130.64bn
dcf_valuations.csv says $130.64bn


### The headline in dollars

Summing the absolute valuation miss across firms puts the result in dollars. Growth only means each
forecast's growth with the actual margin, so this isolates what the model was asked to read.

In [22]:
v["miss_growth"] = (v["ev_perfect_foresight"] * v["err_growth_only"]).abs() / 1e9
v["miss_full"] = (v["ev"] - v["ev_perfect_foresight"]).abs() / 1e9
v["flagged"] = v["ticker"].isin(FLAGGED)

usd = pd.DataFrame({
    "growth only, all": v.groupby("scenario")["miss_growth"].sum(),
    "growth only, ex-flagged": v[~v["flagged"]].groupby("scenario")["miss_growth"].sum(),
    "growth + margin, all": v.groupby("scenario")["miss_full"].sum(),
    "growth + margin, ex-flagged": v[~v["flagged"]].groupby("scenario")["miss_full"].sum(),
}).T[list(fc)]
print("summed |valuation miss|, $bn (53 firms, total perfect-foresight EV "
      f"${v[v['scenario'] == 'AI full']['ev_perfect_foresight'].sum() / 1e9:,.0f}bn)")
print(usd.round(1).to_string())

summed |valuation miss|, $bn (53 firms, total perfect-foresight EV $1,276bn)
scenario                     3-yr average  last year  AI full  AI no MD&A
growth only, all                    133.6      132.1    101.5       101.9
growth only, ex-flagged             106.7      100.9     83.4        77.7
growth + margin, all                403.7      275.3    313.3       309.8
growth + margin, ex-flagged         241.8      141.9    182.1       189.5


In dollars the model's growth forecasts cut the summed miss from $134bn to $101bn against the
three-year average. They also beat "last year" ($132bn) on this measure, because a dollar sum weights
big firms and the bootstrap weights every firm equally. Without the four flagged firms the edge over
the three-year average is $107bn to $83bn, so most of it survives in dollar terms. Once margin is
included, "last year" has the smallest total miss ($275bn against $313bn for the model).

## Part 7: Which check backs each number

Lecture 1 says every reported number needs one of four checks. This is the list for the memo and the
slides.

In [23]:
card = pd.DataFrame([
    ["Growth MAE: 6.5% / 5.6% / 4.8% / 5.3%", "independent method", "Part 4: rebuilt from raw files, matches notebook"],
    ["AI vs 3-yr average, all 53: -1.7pp, reliably better", "independent method", "Part 5: second bootstrap, seed 1, matrix draws"],
    ["AI vs 3-yr average, ex-flagged: -0.9pp, not distinguishable", "independent method", "Part 5: new in this notebook"],
    ["AI vs last year: not distinguishable in any group", "independent method", "Part 5"],
    ["Every EV in dcf_valuations.csv", "closed form", "Part 6: array formula, all 212 rows"],
    ["CAT valuation", "hand calculation", "Part 6: five rounded steps in $bn"],
    ["DCF engine (npv, Gordon, scaling)", "closed form", "dcf notebook Part 2"],
    ["Forecast values", "sanity bounds", "Part 1: growth in [-30%, 40%], margin in [-20%, 50%]"],
    ["Quotes in filing: 89% (97% with close paraphrases)", "independent method", "Part 2: five-word pieces vs script's exact match"],
    ["Backlog flag: model = quantified rule on 53 of 53", "independent method", "Part 2: regex rule vs model"],
    ["No leak: 0 of 53 recalled; AI no closer to actuals than last year", "sanity bounds", "Part 3"],
    ["Numbers in quotes found in context: 94%", "independent method", "Part 2: number plus neighbouring word"],
    ["Summed growth miss: $134bn vs $101bn", "independent method", "Part 6: from the closed-form EVs"],
], columns=["number", "check", "where"])
print(card.to_string(index=False))
card.to_csv(os.path.join(OUT, "verification_scorecard.csv"), index=False)
print("\nsaved output/verification_scorecard.csv")

                                                           number              check                                                where
                            Growth MAE: 6.5% / 5.6% / 4.8% / 5.3% independent method     Part 4: rebuilt from raw files, matches notebook
              AI vs 3-yr average, all 53: -1.7pp, reliably better independent method       Part 5: second bootstrap, seed 1, matrix draws
      AI vs 3-yr average, ex-flagged: -0.9pp, not distinguishable independent method                         Part 5: new in this notebook
                AI vs last year: not distinguishable in any group independent method                                               Part 5
                                   Every EV in dcf_valuations.csv        closed form                  Part 6: array formula, all 212 rows
                                                    CAT valuation   hand calculation                    Part 6: five rounded steps in $bn
                                DC

## What this means for the memo

The model's output is clean, it grounds about nine in ten quotes in the filing, and it can't have seen
the answers. The DCF numbers check out two independent ways. The finding that needs care is how much
the model actually helps: its edge over the three-year average is reliable only when the four firms
with broken histories stay in, and it's never reliably better than repeating last year. The memo
should say that plainly.